In [1]:
import sys
import os

# Obtenemos la ruta de la carpeta actual (/Workspace/.../silver)
current_dir = os.getcwd()
# Subimos un nivel hacia la carpeta raíz de tu proyecto
parent_dir = os.path.dirname(current_dir)

# Agregamos la raíz de tu proyecto al cerebro de Python
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from src.etl_master import LakehouseMaster
from pyspark.sql.functions import col, to_date, year, month, round as spark_round

In [3]:
class FctSales(LakehouseMaster):
    def transform(self, dfs):
        self.logger.info("Calculando métricas de ventas (Fact Table)")
        
        # Como el master detectó "multiple_tables", nos entrega un diccionario
        df_orders = dfs["orders"]
        df_items = dfs["items"]

        # Join entre órdenes y detalle
        fct = df_orders.join(df_items, df_orders.id == df_items.order_id, "inner")

        # Reglas de negocio
        fct_transformed = fct.select(
            col("order_id"),
            col("user_id"),
            col("product_id"),
            to_date(col("order_date")).alias("date_sk"),
            col("status"),
            col("quantity"),
            col("price"),
            spark_round((col("quantity") * col("price")), 2).alias("line_total_amount")
        ).withColumn("part_year", year("date_sk")) \
         .withColumn("part_month", month("date_sk"))
        
        return fct_transformed